# Ordered Logistic Regression Results: FAIR² Dataset Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the [FAIR² dataset](https://sen.science/doi/10.71728/senscience.y7m0-f273) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided as a Croissant schema at the following URL:

`https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)

# Print basic metadata
print(f"Dataset name: {dataset.metadata.name}\nDescription: {dataset.metadata.description}")
print(f"Published: {dataset.metadata.datePublished}")
print(f"License: {dataset.metadata.license}")

## 2. Data Overview
Review the available record sets, their fields, columns, and their unique `@id`s.

Let's list all record sets, and for each, print fields and columns (`@id`). This information is crucial for accessing and extracting the correct data.

In [ ]:
# List available record sets and their fields/columns
record_sets = list(dataset.record_sets)
if not record_sets:
    print("No record sets defined in the metadata.")
else:
    print(f"Found {len(record_sets)} record set(s) in this dataset.")
    for rs in record_sets:
        print(f"\nRecord set name: {rs.name}")
        print(f"  @id: {rs.id}")
        if getattr(rs, 'fields', None):
            print(f"  Fields:")
            for field in rs.fields:
                print(f"    - {field.name} (@id: {field.id}) [type: {field.data_type}]")
        if getattr(rs, 'columns', None):
            print(f"  Columns:")
            for col in rs.columns:
                print(f"    - {col.name} (@id: {col.id}) [type: {col.data_type}]")


## 3. Data Extraction
Load data from the record set(s) into pandas DataFrames for analysis. Use the record set and field `@id`s from the overview above.

We will demonstrate how to extract records for each record set by their `@id`. Adjust the selected IDs as needed for your analysis.

In [ ]:
#--- Step 1: List all available record set IDs from the overview above
available_ids = [rs.id for rs in dataset.record_sets]
print("Available record sets (by @id):", available_ids)

#--- Step 2: Load each record set into a DataFrame ---#
dataframes = {}
for rs_id in available_ids:
    records = list(dataset.records(record_set=rs_id))
    dataframes[rs_id] = pd.DataFrame(records)
    print(f"\nLoaded {len(records)} records for record set '@id': {rs_id}")
    print(f"Columns: {dataframes[rs_id].columns.tolist()}")

#--- Step 3: Inspect one of the DataFrames (choose the first record set if present) ---#
if available_ids:
    sample_rs_id = available_ids[0]
    print(f"\nPreview of records from record set '@id': {sample_rs_id}")
    display(dataframes[sample_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply basic data processing steps: filtering, normalization, and grouping.

_Choose a numeric field (by its `@id`) and a grouping field (`@id`). Adjust the code below to match the fields relevant to your dataset as printed in the previous cells._

In [ ]:
#--- Select example record set, numeric field, and grouping field (adjust as needed) ---#
if available_ids:
    record_set_id = sample_rs_id
    df = dataframes[record_set_id]
    print(f"Working with record set '@id': {record_set_id}")
    print("Columns:", df.columns.tolist())
    
    # Guess the numeric field and group field (edit as appropriate)
    # Example: 'log_likelihood' and 'ward_name' (replace with actual @id/column names)
    available_numeric_columns = df.select_dtypes(include=['number']).columns.tolist()
    if available_numeric_columns:
        numeric_field = available_numeric_columns[0]
        print(f"Using numeric field: {numeric_field}")
    else:
        print("No numeric field detected.")
        numeric_field = None

    # Attempt to use a string column for grouping
    potential_group_cols = [c for c in df.columns if df[c].dtype == 'object']
    group_field = None
    if potential_group_cols:
        group_field = potential_group_cols[0]
        print(f"Grouping by: {group_field}")

    if numeric_field:
        threshold = df[numeric_field].mean() if not pd.isna(df[numeric_field].mean()) else 0
        #--- Filtering above threshold ---#
        filtered_df = df[df[numeric_field] > threshold].copy()
        print(f"Filtered {len(filtered_df)} records with {numeric_field} > {threshold:.2f}.")

        #--- Normalization ---#
        filtered_df[f"{numeric_field}_normalized"] = (
            (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) /
            (filtered_df[numeric_field].std() if filtered_df[numeric_field].std() > 0 else 1)
        )
        print(f"\nFirst 5 records with normalized '{numeric_field}':")
        display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

        #--- Grouping by a categorical field (optional) ---#
        if group_field and group_field in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
            print(f"\nMean {numeric_field} by {group_field}:")
            print(grouped_df.head())
else:
    print("No record sets available to analyze.")

## 5. Visualization
Visualize the distribution of the numeric field, optionally grouped by a categorical variable.

_This cell assumes that a valid DataFrame (`filtered_df`), `numeric_field`, and `group_field` are available from above. Adjust the field names if needed._

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and numeric_field:
    plt.figure(figsize=(8, 5))
    sns.histplot(filtered_df[numeric_field], kde=True)
    plt.title(f"Distribution of {numeric_field}")
    plt.xlabel(numeric_field)
    plt.ylabel('Frequency')
    plt.show()

    if group_field and group_field in filtered_df.columns:
        plt.figure(figsize=(10, 6))
        sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
        plt.xticks(rotation=45)
        plt.title(f"{numeric_field} by {group_field}")
        plt.xlabel(group_field)
        plt.ylabel(numeric_field)
        plt.tight_layout()
        plt.show()
else:
    print("No data available for plotting.")

## 6. Conclusion
In this notebook, we used the `mlcroissant` library to load and explore the FAIR² ordered logistic regression dataset. We demonstrated:

- Retrieving Croissant schema metadata and exploring dataset structure
- Listing record sets, fields, and their `@id`s
- Loading record sets as pandas DataFrames
- Performing basic filtering, normalization, and group-based aggregation on numeric fields
- Visualizing distributions and group-wise differences

Further exploration could include:
- Advanced statistical tests on modeling results
- Visualizing relationships between multiple predictors
- Joining or merging with other datasets using relevant `@id`s.

For more, see the [mlcroissant documentation](https://github.com/mlcommons/croissant).